# Инициализация, оптимизаторы и расписания шага

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

Объяснять роль инициализации, оптимизатора и расписания шага в одном воспроизводимом опыте.

## Начальные веса — часть опыта

Если все нейроны одного слоя начинают одинаково, получают одинаковые входы и одинаковые градиенты, они могут продолжать делать одно и то же. Случайная инициализация разрывает эту симметрию. Масштаб весов также важен: слишком большие или маленькие промежуточные значения затрудняют обучение. Готовые слои PyTorch задают начальные значения по своим правилам.

Seed повторяет источник случайности в одном окружении, но не гарантирует, что один запуск представляет типичное качество. Для сравнения настроек используют одинаковый протокол и несколько заранее выбранных стартов на данных разработки.

## Разобранный пример: меняющийся шаг

Первые 10 шагов используем η=0.1, затем η=0.05. На одном и том же градиенте 2 поправка сначала −0.2, затем −0.1. **Расписание шага** задаёт это изменение по номеру шага или по правилу проверки. Оно не меняет функцию потерь, но меняет путь её оптимизации.

SGD использует текущий градиент, вариант с momentum дополнительно сохраняет накопленное направление движения. Adam хранит движущиеся оценки градиента и его квадрата и по-разному масштабирует координаты. Это алгоритмы обновления, а не способы выбрать хорошую итоговую модель без validation.

Что означает «хранит направление»? В простом варианте momentum начинаем с $v_0=0$ и считаем $v_{t+1}=\beta v_t+g_t$, затем $w_{t+1}=w_t-\eta v_{t+1}$. Здесь t=0,1,2,… — номер шага, $g_t$ — градиент в старой точке $w_t$, $0\leq\beta<1$ — доля прошлого направления. При $\beta=0.5$ и двух градиентах 2 получаем $v_1=2$, $v_2=3$: второй шаг учитывает и прошлое. У Adam своё правило: он хранит сглаженные средние градиентов и квадратов градиентов. Поэтому веса без этих накопленных величин не определяют следующий шаг Adam. Формулы разных оптимизаторов не следует смешивать.

## Согласованная запись

Сохраняйте инициализацию, шаг, число обновлений и порядок объектов. При возобновлении обучения одного состояния весов недостаточно для точного продолжения Adam: важно состояние оптимизатора и расписания. Для простого применения уже обученной модели оно не требуется.

В ноутбуке StepLR уменьшает lr по фиксированному числу эпох. scheduler.step() выполняется после шага оптимизатора; get_last_lr возвращает текущие значения шага. Чтобы опыт был коротким, в коде шаг уменьшается после каждых двух эпох, а не после десяти из ручного примера. При четырёх обновлениях используем [0.1,0.1,0.05,0.05]; после последнего `scheduler.step()` шаг для следующей эпохи уже 0.025. Мы проверяем само расписание, а не выбираем его по тесту.

`optimizer.param_groups` хранит группы параметров с их настройками; в примере группа одна, и поле `lr` показывает фактически использованный шаг.

## Разобранный пример: проверяем вычисление

Все числа в примере придуманы для этого урока. Выполните код, затем измените вход и объясните новый результат.

In [ ]:
import torch
from torch import nn
torch.set_num_threads(1)
torch.manual_seed(17)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
used = []
for _ in range(4):
    used.append(optimizer.param_groups[0]["lr"])
    optimizer.zero_grad(set_to_none=True)
    model(torch.ones(1, 1)).square().mean().backward()
    optimizer.step()
    scheduler.step()
assert used == [0.1, 0.1, 0.05, 0.05]
assert scheduler.get_last_lr() == [0.025]
print(used)


## Типичные ошибки

Не вызывайте scheduler до соответствующего optimizer.step без понимания соглашения. Не обещайте точное продолжение только по весам.

## Задачи

### 1. Базовая

Зачем случайно различать нейроны при старте?

<details>
<summary>Подсказка</summary>

Иначе они могут оставаться копиями.

</details>



### 2. Базовая

Что меняет расписание?

<details>
<summary>Подсказка</summary>

Это не новая метка или потеря.

</details>



### 3. Прикладная

Градиент 4, lr=0.05. Какова поправка SGD?

<details>
<summary>Подсказка</summary>

Вычесть lr·градиент.

</details>



### 4. Прикладная

Шаг 0.1 уменьшили вдвое дважды. Чему он равен?

<details>
<summary>Подсказка</summary>

Умножьте на 0.5².

</details>



### 5. Олимпиадная

Почему для продолжения Adam нужны не только веса?

<details>
<summary>Подсказка</summary>

У него есть накопленные оценки.

</details>



### 6. Олимпиадная

Можно ли выбрать самый удачный seed на тесте и считать оценку независимой?

<details>
<summary>Подсказка</summary>

Seed становится подбираемым параметром.

</details>



## Итоги и самопроверка

Объясните, какие состояния нужны для применения модели и для продолжения обучения. Как проверить фактическое расписание?

Для перехода дальше достаточно базовых и прикладных задач. Олимпиадные задачи можно отложить.


## Решения задач

### Задача 1

Чтобы разорвать симметрию одинаковых вычислений и градиентов.

### Задача 2

Размер шага по ходу обучения.

### Задача 3

−0.2.

### Задача 4

0.025.

### Задача 5

Они определяют следующие обновления. Без них продолжится другой путь обучения.

### Задача 6

Нет. Это такой же отбор по тестовым ответам.